In [64]:
import numpy as np
import pandas as pd
import datetime as dt

In [79]:
games_df = pd.read_csv('data/international_football_games.csv')
draws_df = pd.read_csv('data/shootouts.csv')    
valid_teams_df = pd.read_csv('data/valid_fifa_teams.csv')

In [66]:
print(games_df["tournament"].unique().tolist())

['Friendly', 'British Home Championship', 'Évence Coppée Trophy', 'Muratti Vase', 'Copa Lipton', 'Copa Newton', 'Copa Premio Honor Argentino', 'Olympic Games', 'Copa Premio Honor Uruguayo', 'Far Eastern Championship Games', 'Copa Roca', 'Copa América', 'Inter-Allied Games', 'Peace Cup', 'Open International Championship', 'Soccer Ashes', 'Copa Chevallier Boutell', 'Nordic Championship', 'Central European International Cup', 'Baltic Cup', 'Balkan Cup', 'Central American and Caribbean Games', 'FIFA World Cup', 'Copa Rio Branco', 'FIFA World Cup qualification', 'Bolivarian Games', 'CCCF Championship', 'NAFC Championship', 'Copa Oswaldo Cruz', 'Asian Games', 'Pan American Championship', 'Copa del Pacífico', "Copa Bernardo O'Higgins", 'AFC Asian Cup qualification', 'Atlantic Cup', 'AFC Asian Cup', 'African Cup of Nations', 'Copa Paz del Chaco', 'Merdeka Tournament', 'UEFA Euro qualification', 'Southeast Asian Peninsular Games', 'African Friendship Games', 'UEFA Euro', 'Windward Islands Tourn

In [80]:
home_teams = games_df["home_team"].unique().tolist()
away_teams = games_df["away_team"].unique().tolist()
all_teams = set(home_teams + away_teams)
for team in valid_teams_df["team"]:
    if team not in all_teams:
        print(f"{team} not in any teams")

In [81]:
teams = valid_teams_df["team"].unique().tolist()
print(games_df["date"].min(), games_df["date"].max(), games_df.shape[0])
for row_idx, row in games_df.iterrows():
    if row["home_team"] not in teams or row["away_team"] not in teams:
        games_df.drop(row_idx, inplace=True)
print(games_df["date"].min(), games_df["date"].max(), games_df.shape)


1872-11-30 2026-06-27 49477
1872-11-30 2026-06-27 (45478, 9)


In [76]:
display(games_df.head(10))

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False
5,1876-03-25,Scotland,Wales,4.0,0.0,Friendly,Glasgow,Scotland,False
6,1877-03-03,England,Scotland,1.0,3.0,Friendly,London,England,False
7,1877-03-05,Wales,Scotland,0.0,2.0,Friendly,Wrexham,Wales,False
8,1878-03-02,Scotland,England,7.0,2.0,Friendly,Glasgow,Scotland,False
9,1878-03-23,Scotland,Wales,9.0,0.0,Friendly,Glasgow,Scotland,False


In [82]:
winner = None
games_df=games_df.merge(draws_df, how='left', left_on=['home_team', 'away_team', 'date'], right_on=['home_team', 'away_team', 'date']).drop(columns=['first_shooter'])
games_df["date"] = pd.to_datetime(games_df["date"])

In [83]:
games_df.loc[games_df["home_score"] > games_df["away_score"], "winner"] = games_df["home_team"]
games_df.loc[games_df["home_score"] < games_df["away_score"], "winner"] = games_df["away_team"]
games_df.loc[
    (games_df["home_score"] == games_df["away_score"]) &
    (games_df["winner"].isna()),
    "winner"
] = 'DRAW'

games_df["winner_code"] = np.select(
    [
        games_df["winner"] == games_df["home_team"],
        games_df["winner"] == games_df["away_team"],
        games_df["winner"] == 'DRAW'
    ],
    [1, 2, 0],
)

In [84]:
games_df.to_csv("stored_features/_games_valid_teams.csv", index=False)

In [144]:
games_df = pd.read_csv('stored_features/_games_valid_teams.csv')

In [145]:
print(games_df.iloc[24590:24600])

             date      home_team     away_team  home_score  away_score  \
24590  2003-05-22   South Africa       England         1.0         2.0   
24591  2003-05-24       Botswana        Malawi         1.0         1.0   
24592  2003-05-24          Sudan         Kenya         1.0         1.0   
24593  2003-05-25        Jamaica       Nigeria         3.0         2.0   
24594  2003-05-25        Lesotho      Eswatini         2.0         1.0   
24595  2003-05-25           Togo         Benin         3.0         1.0   
24596  2003-05-26  United States         Wales         2.0         0.0   
24597  2003-05-27       Scotland   New Zealand         1.0         1.0   
24598  2003-05-29        Algeria  Burkina Faso         0.0         1.0   
24599  2003-05-30        Nigeria         Ghana         3.0         1.0   

       tournament       city        country  neutral         winner  \
24590    Friendly     Durban   South Africa    False        England   
24591  COSAFA Cup   Gaborone       Botswana

In [146]:
home = games_df[["date", "home_team", "away_team", "home_score", "away_score", "winner", "neutral"]].rename(
    columns={"home_team": "team", "away_team": "opponent", "home_score": "score", "away_score": "opponent_score", "winner": "winner"}
)
home['is_home'] = 1

away = games_df[["date", "away_team", "home_team", "home_score", "away_score", "winner", "neutral"]].rename(
    columns={"away_team": "team", "home_team": "opponent", "away_score": "score", "home_score": "opponent_score", "winner": "winner"}
)
away['is_home'] = 0
team_features_per_date = pd.concat([home, away]).sort_values(["date", "team"]).reset_index(drop=True)
team_features_per_date["date"] = pd.to_datetime(
    team_features_per_date["date"]
)

In [147]:
team_features_per_date.tail(80)

,date,team,opponent,score,opponent_score,winner,neutral,is_home
90876,2026-06-20,Curaçao,Ecuador,0.0,0.0,DRAW,True,0
90877,2026-06-20,Ecuador,Curaçao,0.0,0.0,DRAW,True,1
90878,2026-06-20,Germany,Ivory Coast,2.0,1.0,Germany,True,1
90879,2026-06-20,Ivory Coast,Germany,1.0,2.0,Germany,True,0
90880,2026-06-20,Japan,Tunisia,4.0,0.0,Japan,True,0
...,...,...,...,...,...,...,...,...
90951,2026-06-27,Ghana,Croatia,NaN,NaN,NaN,True,0
90952,2026-06-27,Jordan,Argentina,NaN,NaN,NaN,True,1
90953,2026-06-27,Panama,England,NaN,NaN,NaN,True,1
90954,2026-06-27,Portugal,Colombia,NaN,NaN,NaN,True,0


In [148]:
def result_points(row):
    if row['team'] == row['winner']:
        return 3
    elif row['opponent'] == row['winner']:
        return 1
    else:
        return 0

team_features_per_date["points"] = team_features_per_date.apply(result_points, axis=1)

In [149]:
team_features_per_date["ppg_last_5"] = (
    team_features_per_date
    .groupby("team")["points"]
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)
team_features_per_date["ppg_last_10"] = (
    team_features_per_date
    .groupby("team")["points"]
    .transform(lambda x: x.shift(1).rolling(10, min_periods=1).mean())
)
team_features_per_date["avg_goals_scored_last_5"] = (
    team_features_per_date
    .groupby("team")["score"]
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)
team_features_per_date["avg_goals_conceded_last_5"] = (
    team_features_per_date
    .groupby("team")["opponent_score"]
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)
team_features_per_date["clean_sheets_rate_last_5"] = (
    team_features_per_date
    .groupby("team")["opponent_score"]
    .transform(lambda x: x.shift(1).eq(0).rolling(5, min_periods=1).mean())
)
team_features_per_date["avg_goal_difference_last_5"] = (
    team_features_per_date
    .groupby("team")[["score", "opponent_score"]]
    .apply(lambda x: (x["score"] - x["opponent_score"]).rolling(5, min_periods=1).mean())
    .reset_index(level=0, drop=True)
)
team_features_per_date["avg_goal_difference_last_10"] = (
    team_features_per_date
    .groupby("team")[["score", "opponent_score"]]
    .apply(lambda x: (x["score"] - x["opponent_score"]).rolling(10, min_periods=1).mean())
    .reset_index(level=0, drop=True)
)
team_features_per_date["btts"] = (
    (team_features_per_date["score"] > 0) &
    (team_features_per_date["opponent_score"] > 0)
)
team_features_per_date["btts_rate_last_5"] = (
    team_features_per_date
        .groupby("team")["btts"]
        .apply(lambda x: x.shift(1).eq(True).rolling(5, min_periods=1).mean())
    .reset_index(level=0, drop=True)
)
team_features_per_date["failed_score_rate_last_5"] = (
    team_features_per_date
    .groupby("team")["score"]
    .transform(lambda x: x.shift(1).eq(0).rolling(5, min_periods=1).mean())
)
team_features_per_date["over_2_5_rate_last_5"] = (
    team_features_per_date
    .groupby("team")["score"]
    .transform(lambda x: x.shift(1).gt(2.5).rolling(5, min_periods=1).mean())
)
team_features_per_date["over_3_5_rate_last_5"] = (
    team_features_per_date
    .groupby("team")["score"]
    .transform(lambda x: x.shift(1).gt(3.5).rolling(5, min_periods=1).mean())
)
team_features_per_date["under_1_5_rate_last_5"] = (
    team_features_per_date
    .groupby("team")["score"]
    .transform(lambda x: x.shift(1).lt(1.5).rolling(5, min_periods=1).mean())
)
team_features_per_date["days_since_last_game"] = (
    team_features_per_date
    .groupby("team")["date"]
    .diff()
    .dt.days
    .fillna(0)
)

team_features_per_date["goal_difference_trend"] = team_features_per_date["avg_goal_difference_last_5"] - team_features_per_date["avg_goal_difference_last_10"]
team_features_per_date["momentum"] = team_features_per_date["ppg_last_5"] - team_features_per_date["ppg_last_10"]


In [150]:
team_features_per_date.columns

Index(['date', 'team', 'opponent', 'score', 'opponent_score', 'winner',
       'neutral', 'is_home', 'points', 'ppg_last_5', 'ppg_last_10',
       'avg_goals_scored_last_5', 'avg_goals_conceded_last_5',
       'clean_sheets_rate_last_5', 'avg_goal_difference_last_5',
       'avg_goal_difference_last_10', 'btts', 'btts_rate_last_5',
       'failed_score_rate_last_5', 'over_2_5_rate_last_5',
       'over_3_5_rate_last_5', 'under_1_5_rate_last_5', 'days_since_last_game',
       'goal_difference_trend', 'momentum'],
      dtype='str')

In [151]:
def add_pi_ratings(team_features_per_date, params):
    """
    Adds Pi ratings to a team-per-row dataframe.

    Required columns
    ----------------
    date
    team
    opponent
    score
    opponent_score
    neutral (bool)

    Returns
    -------
    dataframe with additional Pi columns.
    """

    df = team_features_per_date.copy()
    df = df.sort_values("date").reset_index(drop=True)

    c, mu1, mu2 = params
    #c - Controls how ratings are scaled based on the goal difference.
    #mu1 - Controls how much the current match contributes to the ratings.
    #mu2 - Controls how much the secondary rating is adjusted based on the current match's result.

    teams = pd.unique(
        pd.concat([df["team"], df["opponent"]], ignore_index=True)
    )

    pi = {}

    for t in teams:
        pi[f"Home {t}"] = 0.0
        pi[f"Away {t}"] = 0.0

    home_key = "Home {}"
    away_key = "Away {}"

    # ------------------------------------------------------------------
    # Pi helper functions
    # ------------------------------------------------------------------

    def exp_goal_diff(c, hr, ar):

        if ar >= 0:
            egda = 10 ** (abs(ar) / c) - 1
        else:
            egda = -(10 ** (abs(ar) / c) - 1)

        if hr >= 0:
            egdh = 10 ** (abs(hr) / c) - 1
        else:
            egdh = -(10 ** (abs(hr) / c) - 1)

        return egdh - egda

    def weighted_error(c, obs, exp):

        err = abs(obs - exp)

        if exp < obs:
            w1 = c * np.log10(1 + err)
            w2 = -w1
        else:
            w1 = -c * np.log10(1 + err)
            w2 = -w1

        return w1, w2

    # ------------------------------------------------------------------

    out = []

    processed = set()

    for idx, row in df.iterrows():

        if idx in processed:
            continue

        mask = (
            (df["date"] == row["date"])
            & (
                ((df["team"] == row["team"]) & (df["opponent"] == row["opponent"]))
                |
                (
                    (df["team"] == row["opponent"])
                    & (df["opponent"] == row["team"])
                )
            )
        )

        match = df[mask]

        if len(match) != 2:
            continue

        i1, i2 = match.index
        processed.add(i1)
        processed.add(i2)

        r1 = df.loc[i1]
        r2 = df.loc[i2]

        # --------------------------------------------------------------
        # Decide who is home
        # --------------------------------------------------------------

        if r1["neutral"]:

            # Both teams are treated as away teams
            teamA = r1["team"]
            teamB = r2["team"]

            A_rating = pi[away_key.format(teamA)]
            B_rating = pi[away_key.format(teamB)]

            egd = exp_goal_diff(c, A_rating, B_rating)

            obs = r1["score"] - r2["score"]

            wA, wB = weighted_error(c, obs, egd)

            newA = A_rating + wA * mu1
            newB = B_rating + wB * mu1

            pi[away_key.format(teamA)] = newA
            pi[away_key.format(teamB)] = newB

            # Home ratings remain unchanged

            extra1 = {
                "pi_home_rating": pi[home_key.format(teamA)],
                "pi_away_rating": A_rating,
                "pi_expected_gd": egd,
                "pi_diff": A_rating - B_rating,
            }

            extra2 = {
                "pi_home_rating": pi[home_key.format(teamB)],
                "pi_away_rating": B_rating,
                "pi_expected_gd": -egd,
                "pi_diff": B_rating - A_rating,
            }

        else:

            # One row must represent the home team.
            # Here we assume a boolean column named is_home.
            if r1["is_home"]:
                home = r1
                away = r2
                home_idx = i1
                away_idx = i2
            else:
                home = r2
                away = r1
                home_idx = i2
                away_idx = i1

            h_hr = pi[home_key.format(home["team"])]
            h_ar = pi[away_key.format(home["team"])]

            a_hr = pi[home_key.format(away["team"])]
            a_ar = pi[away_key.format(away["team"])]

            egd = exp_goal_diff(c, h_hr, a_ar)

            obs = home["score"] - away["score"]

            w_home, w_away = weighted_error(c, obs, egd)

            h_hr_new = h_hr + w_home * mu1
            h_ar_new = h_ar + (h_hr_new - h_hr) * mu2

            a_ar_new = a_ar + w_away * mu1
            a_hr_new = a_hr + (a_ar_new - a_ar) * mu2

            pi[home_key.format(home["team"])] = h_hr_new
            pi[away_key.format(home["team"])] = h_ar_new

            pi[home_key.format(away["team"])] = a_hr_new
            pi[away_key.format(away["team"])] = a_ar_new

            extra_home = {
                "pi_home_rating": h_hr,
                "pi_away_rating": h_ar,
                "pi_expected_gd": egd,
                "pi_diff": h_hr - a_ar,
            }

            extra_away = {
                "pi_home_rating": a_hr,
                "pi_away_rating": a_ar,
                "pi_expected_gd": -egd,
                "pi_diff": a_ar - h_hr,
            }

            extra1 = extra_home if home_idx == i1 else extra_away
            extra2 = extra_home if home_idx == i2 else extra_away

        out.append((i1, extra1))
        out.append((i2, extra2))

    extras = pd.DataFrame(
        [v for _, v in sorted(out, key=lambda x: x[0])],
        index=[i for i, _ in sorted(out, key=lambda x: x[0])],
    )

    df = df.join(extras)

    return df

In [152]:
c, mu1, mu2 = 10, 0.40, 0.20
params = (
    c,    # c
    mu1,   # mu1
    mu2    # mu2
)

df = add_pi_ratings(team_features_per_date, params)

In [153]:
df.to_csv('stored_features/team_features_c_3_mu1_0.50_mu2_0.20.csv', index=False)

In [154]:
team_features = pd.read_csv('stored_features/team_features_c_3_mu1_0.50_mu2_0.20.csv')

In [155]:
df = team_features.copy()
home_ = df.add_prefix("home_")
away_ = df.add_prefix("away_")
home_.drop(columns=[  "home_opponent_score", "home_winner", "home_is_home", "home_points",], inplace=True)
away_.drop(columns=[ "away_opponent_score", "away_winner", "away_is_home", "away_points","away_neutral"], inplace=True)

In [156]:
matches = home_.merge(
    away_,
    left_on=["home_date", "home_team", "home_opponent"],
    right_on=["away_date", "away_opponent", "away_team"],
    how="inner",
)
matches = matches[
    matches["home_team"] < matches["away_team"]
].reset_index(drop=True)
matches.drop(columns=["away_date", "away_opponent", "away_team"], inplace=True)
matches = matches.rename(columns={"home_opponent": "away_team", "home_date": "date"})

In [157]:
matches.loc[matches["home_score"] > matches["away_score"], "winner"] = matches["home_team"]
matches.loc[matches["home_score"] < matches["away_score"], "winner"] = matches["away_team"]
matches.loc[
    (matches["home_score"] == matches["away_score"]) &
    (matches["winner"].isna()),
    "winner"
] = 'DRAW'

matches["winner_code"] = np.select(
    [
        matches["winner"] == matches["home_team"],
        matches["winner"] == matches["away_team"],
        matches["winner"] == 'DRAW'
    ],
    [1, 2, 0],
)

In [158]:
matches = matches[['date', 'home_team', 'away_team', 'home_score', 'away_score','winner','winner_code','home_neutral',
                   
       'home_ppg_last_5', 'home_ppg_last_10', 'home_avg_goals_scored_last_5',
       'home_avg_goals_conceded_last_5', 'home_clean_sheets_rate_last_5',
       'home_avg_goal_difference_last_5', 'home_avg_goal_difference_last_10',
       'home_btts', 'home_btts_rate_last_5', 'home_failed_score_rate_last_5',
       'home_over_2_5_rate_last_5', 'home_over_3_5_rate_last_5',
       'home_under_1_5_rate_last_5', 'home_days_since_last_game',
       'home_goal_difference_trend', 'home_momentum', 'home_pi_home_rating',
       'home_pi_away_rating', 'home_pi_expected_gd', 'home_pi_diff',
       
       'away_ppg_last_5', 'away_ppg_last_10',
       'away_avg_goals_scored_last_5', 'away_avg_goals_conceded_last_5',
       'away_clean_sheets_rate_last_5', 'away_avg_goal_difference_last_5',
       'away_avg_goal_difference_last_10', 'away_btts',
       'away_btts_rate_last_5', 'away_failed_score_rate_last_5',
       'away_over_2_5_rate_last_5', 'away_over_3_5_rate_last_5',
       'away_under_1_5_rate_last_5', 'away_days_since_last_game',
       'away_goal_difference_trend', 'away_momentum', 'away_pi_home_rating',
       'away_pi_away_rating', 'away_pi_expected_gd', 'away_pi_diff']]

In [159]:
matches.loc[matches["home_neutral"] == 1, "pi_rating_diff"] = matches["home_pi_away_rating"] - matches["away_pi_away_rating"]
matches.loc[matches["home_neutral"] == 0, "pi_rating_diff"] = matches["home_pi_home_rating"] - matches["away_pi_away_rating"]

matches["diff_ppg_last_5"] = matches["home_ppg_last_5"] - matches["away_ppg_last_5"]
matches["diff_goal_difference"] = matches["home_avg_goal_difference_last_5"] - matches["away_avg_goal_difference_last_5"]
matches["diff_btts_rate"] = matches["home_btts_rate_last_5"] - matches["away_btts_rate_last_5"]
matches["diff_avg_goals_scored_last_5"] = matches["home_avg_goals_scored_last_5"] - matches["away_avg_goals_scored_last_5"]
matches["diff_avg_goals_conceded_last_5"] = matches["home_avg_goals_conceded_last_5"] - matches["away_avg_goals_conceded_last_5"]
matches["diff_clean_sheets_rate"] = matches["home_clean_sheets_rate_last_5"] - matches["away_clean_sheets_rate_last_5"]
matches["diff_over_2_5_rate"] = matches["home_over_2_5_rate_last_5"] - matches["away_over_2_5_rate_last_5"]
matches["diff_under_1_5_rate"] = matches["home_under_1_5_rate_last_5"] - matches["away_under_1_5_rate_last_5"]
matches["diff_rests_days"] = matches["home_days_since_last_game"] - matches["away_days_since_last_game"]

# matches.loc[matches["winner_code"] != 0, "can_draw"] = 0
# matches.loc[matches["winner_code"] == 0, "can_draw"] = 1



In [160]:
matches.tail(60)

,date,home_team,away_team,home_score,away_score,winner,winner_code,home_neutral,home_ppg_last_5,home_ppg_last_10,...,pi_rating_diff,diff_ppg_last_5,diff_goal_difference,diff_btts_rate,diff_avg_goals_scored_last_5,diff_avg_goals_conceded_last_5,diff_clean_sheets_rate,diff_over_2_5_rate,diff_under_1_5_rate,diff_rests_days
45450,2026-06-15,Saudi Arabia,Uruguay,1.0,1.0,DRAW,0,True,1.2,1.7,...,-3.312943,0.4,1.00,-0.2,0.2,0.2,0.0,0.2,0.0,-70.0
45451,2026-06-15,Cape Verde,Spain,0.0,0.0,DRAW,0,True,1.8,1.9,...,-4.056550,0.6,-1.40,0.0,-0.4,1.0,-0.2,-0.2,0.2,2.0
45452,2026-06-15,Belgium,Egypt,1.0,1.0,DRAW,0,True,2.4,2.1,...,1.788234,0.8,1.20,0.2,2.8,0.2,-0.2,0.4,-0.6,0.0
45453,2026-06-15,Iran,New Zealand,2.0,2.0,DRAW,0,True,2.2,2.2,...,2.435688,0.8,2.40,0.2,1.4,-1.4,0.6,0.2,-0.4,2.0
45454,2026-06-16,Iraq,Norway,1.0,4.0,Norway,2,True,1.6,1.8,...,-4.440576,0.2,-1.40,-0.4,-1.0,0.0,0.0,-0.4,0.2,-2.0
45455,2026-06-16,France,Senegal,3.0,1.0,France,1,True,2.6,2.5,...,3.002106,1.0,1.00,0.6,1.0,-0.2,-0.4,0.4,-0.2,1.0
45456,2026-06-16,Algeria,Argentina,0.0,3.0,Argentina,2,True,2.0,2.3,...,-0.505653,-1.0,-1.00,-0.2,-0.4,0.2,0.0,0.0,0.6,-1.0
45457,2026-06-16,Austria,Jordan,3.0,1.0,Austria,1,True,2.4,2.5,...,2.871587,1.8,3.00,-0.4,0.6,-2.2,0.6,0.2,0.2,6.0
45458,2026-06-17,Colombia,Uzbekistan,3.0,1.0,Colombia,1,True,2.2,2.3,...,3.388577,0.6,1.20,0.0,0.8,-0.2,0.2,0.2,-0.2,1.0
45459,2026-06-17,Croatia,England,2.0,4.0,England,2,True,2.2,2.3,...,-3.813997,0.2,-1.80,0.6,0.2,1.4,-0.6,0.0,-0.2,3.0


In [161]:
matches.columns

Index(['date', 'home_team', 'away_team', 'home_score', 'away_score', 'winner',
       'winner_code', 'home_neutral', 'home_ppg_last_5', 'home_ppg_last_10',
       'home_avg_goals_scored_last_5', 'home_avg_goals_conceded_last_5',
       'home_clean_sheets_rate_last_5', 'home_avg_goal_difference_last_5',
       'home_avg_goal_difference_last_10', 'home_btts',
       'home_btts_rate_last_5', 'home_failed_score_rate_last_5',
       'home_over_2_5_rate_last_5', 'home_over_3_5_rate_last_5',
       'home_under_1_5_rate_last_5', 'home_days_since_last_game',
       'home_goal_difference_trend', 'home_momentum', 'home_pi_home_rating',
       'home_pi_away_rating', 'home_pi_expected_gd', 'home_pi_diff',
       'away_ppg_last_5', 'away_ppg_last_10', 'away_avg_goals_scored_last_5',
       'away_avg_goals_conceded_last_5', 'away_clean_sheets_rate_last_5',
       'away_avg_goal_difference_last_5', 'away_avg_goal_difference_last_10',
       'away_btts', 'away_btts_rate_last_5', 'away_failed_score_rate

In [162]:
matches = matches.drop(columns=["home_btts", "away_btts", "home_ppg_last_10", "away_ppg_last_10",
                                "home_btts_rate_last_5", "away_btts_rate_last_5","home_goal_difference_trend", "away_goal_difference_trend","home_momentum", "away_momentum", "diff_clean_sheets_rate"])

In [163]:
matches.to_csv('stored_features/match_features_c_3_mu1_0.50_mu2_0.20.csv', index=False)

In [164]:
latest_team_ratings = team_features_per_date.sort_values("date").groupby("team").tail(1).reset_index(drop=True)
display(latest_team_ratings)

,date,team,opponent,score,opponent_score,winner,neutral,is_home,points,ppg_last_5,...,avg_goal_difference_last_10,btts,btts_rate_last_5,failed_score_rate_last_5,over_2_5_rate_last_5,over_3_5_rate_last_5,under_1_5_rate_last_5,days_since_last_game,goal_difference_trend,momentum
0,2019-09-10,Eritrea,Namibia,0.0,2.0,Namibia,False,0,1,1.0,...,-1.900000,False,0.4,0.6,0.0,0.0,1.0,6.0,0.300000,0.1
1,2024-09-06,Cook Islands,Tonga,1.0,3.0,Tonga,True,1,1,1.4,...,-2.800000,True,0.0,0.8,0.0,0.0,1.0,164.0,1.200000,0.0
2,2024-09-09,Tonga,Samoa,1.0,2.0,Samoa,False,0,1,1.8,...,-3.100000,True,0.8,0.2,0.4,0.2,0.6,3.0,3.300000,0.4
3,2024-11-18,Samoa,New Zealand,0.0,8.0,New Zealand,False,0,1,1.8,...,-1.800000,False,0.6,0.2,0.0,0.0,0.6,3.0,-0.400000,-0.2
4,2025-03-21,Tahiti,New Caledonia,0.0,3.0,New Caledonia,True,0,1,2.2,...,-0.200000,False,0.2,0.4,0.2,0.0,0.4,123.0,0.200000,0.1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
206,2026-06-27,Argentina,Jordan,NaN,NaN,NaN,True,0,0,3.0,...,2.777778,False,0.0,0.0,0.6,0.2,0.0,5.0,-0.277778,0.2
207,2026-06-27,Algeria,Austria,NaN,NaN,NaN,True,1,0,2.0,...,1.222222,False,0.2,0.4,0.2,0.2,0.6,5.0,-0.472222,-0.3
208,2026-06-27,Portugal,Colombia,NaN,NaN,NaN,True,0,0,2.4,...,1.666667,False,0.6,0.0,0.2,0.2,0.2,4.0,0.083333,0.5
209,2026-06-27,Croatia,Ghana,NaN,NaN,NaN,True,1,0,1.8,...,0.333333,False,0.6,0.2,0.0,0.0,0.6,4.0,-0.833333,-0.3
